# ROC Curve and AUC: Derivation and Rank-Sum Interpretation

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/roc-auc)

We construct the ROC curve step by step from sorted scores, verify the Wilcoxon-Mann-Whitney rank-sum interpretation numerically, and compare ROC-AUC vs PR-AUC on class-imbalanced data.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn import metrics
plt.style.use('dark_background')
rng = np.random.default_rng(42)

## 1 — Building the ROC curve from scratch

In [ ]:
def roc_curve_manual(y_true, y_score):
    """Returns (fpr, tpr) arrays for the ROC curve."""
    order = np.argsort(y_score)[::-1]    # sort by descending score
    y_true_sorted = y_true[order]
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    fpr, tpr = [0.], [0.]
    fp = tp = 0
    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1
        fpr.append(fp / n_neg)
        tpr.append(tp / n_pos)
    return np.array(fpr), np.array(tpr)

def auc_trapezoidal(fpr, tpr):
    return np.trapz(tpr, fpr)

# Worked example from the wiki
y_true = np.array([1, 1, 0, 1, 0])
y_score = np.array([0.92, 0.85, 0.61, 0.43, 0.18])

fpr_m, tpr_m = roc_curve_manual(y_true, y_score)
auc_m = auc_trapezoidal(fpr_m, tpr_m)
print(f'Manual AUC = {auc_m:.4f}  (expected 5/6 ≈ {5/6:.4f})')

# Verify against sklearn
fpr_sk, tpr_sk, _ = metrics.roc_curve(y_true, y_score)
auc_sk = metrics.roc_auc_score(y_true, y_score)
print(f'Sklearn AUC = {auc_sk:.4f}')

plt.figure(figsize=(6, 5))
plt.plot(fpr_m, tpr_m, 'o-', color='#6366f1', label=f'ROC (AUC={auc_m:.3f})')
plt.plot([0, 1], [0, 1], '--', color='#6b7280', label='Random (AUC=0.5)')
plt.fill_between(fpr_m, tpr_m, alpha=0.2, color='#6366f1')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve — Worked Example'); plt.legend()
plt.tight_layout(); plt.show()

## 2 — AUC = Wilcoxon-Mann-Whitney rank sum statistic

In [ ]:
def auc_as_rank_sum(y_true, y_score):
    """AUC = fraction of (positive, negative) pairs where positive scores higher."""
    pos_scores = y_score[y_true == 1]
    neg_scores = y_score[y_true == 0]
    concordant = sum(p > n for p in pos_scores for n in neg_scores)
    ties       = sum(p == n for p in pos_scores for n in neg_scores)
    total = len(pos_scores) * len(neg_scores)
    return (concordant + 0.5 * ties) / total

rng_big = np.random.default_rng(0)
y_true_big = rng_big.binomial(1, 0.4, 200)
# Simulate a classifier that's better than random
y_score_big = rng_big.beta(2 if y_true_big.astype(bool) else 1,
                            1 if y_true_big.astype(bool) else 2, 200)
y_score_big = np.where(y_true_big == 1,
                        rng_big.beta(3, 2, 200),
                        rng_big.beta(2, 3, 200))

auc_curve = metrics.roc_auc_score(y_true_big, y_score_big)
auc_ranks  = auc_as_rank_sum(y_true_big, y_score_big)
print(f'AUC from ROC curve = {auc_curve:.5f}')
print(f'AUC from rank sum  = {auc_ranks:.5f}')
print(f'Difference = {abs(auc_curve - auc_ranks):.2e}  (should be ~0)')

## 3 — ROC-AUC vs PR-AUC under class imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for prevalence, color in [(0.5, '#6366f1'), (0.05, '#f87171'), (0.01, '#f59e0b')]:
    n = 2000
    y_true_imb = (rng.uniform(size=n) < prevalence).astype(int)
    # Classifier that separates with moderate quality
    y_score_imb = np.where(y_true_imb == 1,
                            rng.beta(3, 2, n),
                            rng.beta(2, 3, n))
    roc_auc = metrics.roc_auc_score(y_true_imb, y_score_imb)
    pr_auc  = metrics.average_precision_score(y_true_imb, y_score_imb)

    fpr_, tpr_, _ = metrics.roc_curve(y_true_imb, y_score_imb)
    pre_, rec_, _ = metrics.precision_recall_curve(y_true_imb, y_score_imb)

    axes[0].plot(fpr_, tpr_, color=color, label=f'prev={prevalence:.0%} AUC-ROC={roc_auc:.2f}')
    axes[1].plot(rec_, pre_, color=color, label=f'prev={prevalence:.0%} AUC-PR={pr_auc:.2f}')

for ax, title in zip(axes, ['ROC Curve', 'Precision-Recall Curve']):
    ax.plot([0,1],[0,1],'--',color='gray',alpha=0.5)
    ax.set_xlabel(ax.get_xlabel() or 'Recall' if 'PR' in title else 'FPR')
    ax.legend(); ax.set_title(title)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
plt.suptitle('ROC-AUC barely drops under imbalance; PR-AUC exposes the difference')
plt.tight_layout(); plt.show()

## 4 — Threshold selection

In [ ]:
# Youden's J: argmax(TPR - FPR)
y_t = rng.binomial(1, 0.3, 500)
y_s = np.where(y_t, rng.beta(3, 2, 500), rng.beta(2, 3, 500))

fpr_, tpr_, thresholds = metrics.roc_curve(y_t, y_s)
youden_idx = np.argmax(tpr_ - fpr_)
opt_threshold = thresholds[youden_idx]
print(f"Youden's J optimal threshold: {opt_threshold:.3f}")
print(f"  TPR = {tpr_[youden_idx]:.3f},  FPR = {fpr_[youden_idx]:.3f}")

plt.figure(figsize=(6, 5))
plt.plot(fpr_, tpr_, color='#6366f1', label='ROC')
plt.scatter(fpr_[youden_idx], tpr_[youden_idx], color='#f59e0b', zorder=5, s=100,
            label=f'Youden J (τ={opt_threshold:.2f})')
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend()
plt.title("Youden's J: Maximize TPR - FPR"); plt.tight_layout(); plt.show()

## ✏️ Your turn

**Task A — Perfect and random classifiers:** Generate scores for a perfect classifier (positives always score 1, negatives always score 0) and a random one (uniform scores). Plot both ROC curves on the same axes and verify AUC = 1.0 and AUC = 0.5.

**Task B — Cost-sensitive threshold:** If a false negative costs $c_{FN} = 10$ (miss a fraud case) and a false positive costs $c_{FP} = 1$ (alert a legitimate transaction), find the threshold $\tau^* = \arg\min_\tau(c_{FN} \cdot FN + c_{FP} \cdot FP)$ on a test set of 1000 transactions with 5% fraud.

In [ ]:
# Task A
n_a = 200
y_true_a = rng.binomial(1, 0.4, n_a)
# TODO(you): create y_perfect and y_random; plot ROC curves; verify AUCs

# Task B
c_fn, c_fp = 10, 1
n_b = 1000
y_true_b = (rng.uniform(size=n_b) < 0.05).astype(int)
y_score_b = np.where(y_true_b, rng.beta(4, 2, n_b), rng.beta(2, 4, n_b))
# TODO(you): find cost-optimal threshold using roc_curve thresholds
# Hint: cost(tau) = c_fn * FN(tau) + c_fp * FP(tau)

<details><summary>Solution — Task A</summary>

```python
y_perfect = np.where(y_true_a == 1, 1.0, 0.0)
y_random  = rng.uniform(size=n_a)

plt.figure(figsize=(6,5))
for y_s, label in [(y_perfect, 'Perfect'), (y_random, 'Random')]:
    fpr_, tpr_, _ = metrics.roc_curve(y_true_a, y_s)
    auc_ = metrics.roc_auc_score(y_true_a, y_s)
    plt.plot(fpr_, tpr_, label=f'{label} AUC={auc_:.2f}')
plt.plot([0,1],[0,1],'--',color='gray'); plt.legend(); plt.show()
```
</details>

<details><summary>Solution — Task B</summary>

```python
fpr_b, tpr_b, thresholds_b = metrics.roc_curve(y_true_b, y_score_b)
n_pos_b, n_neg_b = y_true_b.sum(), (1-y_true_b).sum()
costs = []
for fpr_val, tpr_val in zip(fpr_b, tpr_b):
    FN = n_pos_b * (1 - tpr_val)
    FP = n_neg_b * fpr_val
    costs.append(c_fn * FN + c_fp * FP)
best_idx = np.argmin(costs)
print(f'Optimal threshold: {thresholds_b[best_idx]:.3f}, cost: {costs[best_idx]:.1f}')
```
</details>